# Import thư viện, Khởi tạo GRU và Metric

In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
class TimeSeriesGRUDataset(Dataset):
    def __init__(self, X, y):
        # X: (số mẫu, 4 phase, số feature mỗi phase)
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # Trả về 1 chuỗi 4 phase + nhãn tương ứng
        return self.X[idx], self.y[idx]

class TimeSeriesGRU(nn.Module):
    def __init__(
        self,
        input_dim_per_phase,  # số feature mỗi phase
        hidden_dim=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3,
        bidirectional=False
    ):
        super().__init__()

        # GRU xử lý chuỗi 4 phase
        self.gru = nn.GRU(
            input_size=input_dim_per_phase,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )

        # Nếu BiGRU thì hidden_dim * 2
        multiplier = 2 if bidirectional else 1

        # Fully connected để phân loại
        self.fc = nn.Linear(hidden_dim * multiplier, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, 4, features)
        out, hn = self.gru(x)

        # Lấy output của phase cuối (phase 4)
        out = out[:, -1, :]
        out = self.dropout(out)

        return self.fc(out)


In [3]:
# Metrics
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0

def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

def print_results(version_name, phase, y_true, y_pred):
    target_names = ['Excellent', 'Good', 'Average']
    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(y_true, y_pred, digits=10, target_names=target_names))

    # G-Mean per class
    print("G-Mean per class (one-vs-rest):")
    for idx, name in enumerate(target_names):
        g = gmean_per_class(y_true, y_pred, idx)
        print(f"  {name:<10}: {g:.10f}")
        
    print()
    
    metrics = {
        'Version': version_name,
        'Phase': phase,
        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),
        'PrecMacro': precision_score(y_true, y_pred, average='macro'),
        'PrecWeighted': precision_score(y_true, y_pred, average='weighted'),
        'RecMacro': recall_score(y_true, y_pred, average='macro'),
        'RecWeighted': recall_score(y_true, y_pred, average='weighted'),
        'F1Macro': f1_score(y_true, y_pred, average='macro'),
        'F1Weighted': f1_score(y_true, y_pred, average='weighted'),
        'GMean': gmean_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred)
    }
    
    for k, v in metrics.items():
        if k not in ['Version', 'Phase']:
            print(f"{k:18} : {v:.10f}")
    
    return metrics

# Hàm chuẩn bị dữ liệu và train

In [4]:
def prepare_and_train_gru(train_path, val_path=None, test_paths=None):
    print(f"Loading train: {train_path}")
    df_train = pd.read_csv(train_path)
    
    # Nếu có validation thì gộp train + val để train chung
    if val_path:
        df_val = pd.read_csv(val_path)
        df = pd.concat([df_train, df_val], ignore_index=True)
        print(f"Combined train + val: {len(df)} samples")
    else:
        df = df_train
        print(f"Only train: {len(df)} samples")
    
    df = df.drop(columns=['user_id', 'course_id'], errors='ignore')
    
    # Tách nhãn
    y = df['label_3'].values
    df_features = df.drop('label_3', axis=1)
    
    # Lấy các cột thuộc 4 phase (_1, _2, _3, _4)
    phase_cols = [col for col in df_features.columns if col.endswith(('_1', '_2', '_3', '_4'))]
    
    # Tên feature gốc (không bao gồm hậu tố phase)
    base_names = sorted(set(col.rsplit('_', 1)[0] for col in phase_cols))
    
    # Tạo dữ liệu chuỗi thời gian (4 phase)
    phases_data = []
    for p in ['1', '2', '3', '4']:
        cols_p = [col for col in phase_cols if col.endswith(f'_{p}')]
        phases_data.append(df_features[cols_p].values)
    
    # Shape: (samples, 4 phase, số feature mỗi phase)
    X = np.stack(phases_data, axis=1)
    
    print(f"X shape: {X.shape} → (samples, seq_len=4, features_per_phase)")
    
    # Chuẩn hóa dữ liệu
    scaler = StandardScaler()
    N, T, F = X.shape
    
    # Scale theo feature
    X_reshaped = X.reshape(-1, F)
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(N, T, F)

    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    print(f"Classes: {le.classes_} → {le.transform(le.classes_)}")

    dataset = TimeSeriesGRUDataset(X, y_enc)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    model = TimeSeriesGRU(
        input_dim_per_phase=F,
        hidden_dim=64,
        num_layers=2,
        dropout=0.3
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Huấn luyện mô hình (có early stopping)
    model.train()
    best_loss = float('inf')
    patience = 10
    wait = 0
    
    for epoch in range(80):
        epoch_loss = 0
        
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        epoch_loss /= len(loader)
        
        # Lưu model tốt nhất
        if epoch_loss < best_loss - 1e-4:
            best_loss = epoch_loss
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    model.load_state_dict(best_state)
    model.eval()
    print("Training done")
    
    return model, scaler, le, base_names

# Chạy từng version

In [5]:
base_path = "/kaggle/input/mooccubex-data-cleaned/split_data"

In [6]:
def run_experiment(
    base_path,
    data_folder,
    train_file,
    val_file,
    test_prefix,
    version_name
):
    print(f"\n{'#'*5}")
    print(f"Version: {version_name}")
    print(f"{'#'*5}")

    train_path = f"{base_path}/{data_folder}/{train_file}"
    val_path   = f"{base_path}/{data_folder}/{val_file}"

    test_files = [
        f"{base_path}/{data_folder}/{test_prefix}_1.csv",
        f"{base_path}/{data_folder}/{test_prefix}_2.csv",
        f"{base_path}/{data_folder}/{test_prefix}_3.csv",
        f"{base_path}/{data_folder}/{test_prefix}_4.csv",
    ]

    # Huấn luyện mô hình GRU
    model, scaler, le, _ = prepare_and_train_gru(train_path, val_path)
    results = []
    model.eval()
    with torch.no_grad():

        # Đánh giá mô hình trên từng phase test
        for phase, test_path in enumerate(test_files, 1):
            print(f"\n--- Test Phase {phase}: {test_path} ---")

            df_test = pd.read_csv(test_path)

            df_test = df_test.drop(columns=['user_id', 'course_id'], errors='ignore')

            # Tách nhãn
            y_test = df_test['label_3'].values
            df_test = df_test.drop('label_3', axis=1)

            # Tách đặc trưng theo 4 phase
            phase_cols = [
                col for col in df_test.columns
                if col.endswith(('_1', '_2', '_3', '_4'))
            ]

            phases_data = []
            for p in ['1', '2', '3', '4']:
                cols_p = [c for c in phase_cols if c.endswith(f'_{p}')]
                phases_data.append(df_test[cols_p].values)

            # Dạng dữ liệu cuối: (số mẫu, seq_len=4, số đặc trưng mỗi phase)
            X_test = np.stack(phases_data, axis=1)

            # Chuẩn hóa dữ liệu (Dùng scaler đã fit từ tập train)
            N, T, F = X_test.shape
            X_test_scaled = scaler.transform(
                X_test.reshape(-1, F)
            ).reshape(N, T, F)

            # Dự đoán bằng mô hình GRU
            X_tensor = torch.FloatTensor(X_test_scaled).to(device)
            preds = torch.argmax(model(X_tensor), dim=1).cpu().numpy()

            # Encode nhãn
            y_test_enc = le.transform(y_test)

            res = print_results(version_name, phase, y_test_enc, preds)
            results.append(res)

    return pd.DataFrame(results).round(10)

## V0 (Raw Median)

In [7]:
df_v0 = run_experiment(
    base_path=base_path,
    data_folder="data_median_im_3",
    train_file="train_us.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V0 (Raw Median)"
)

df_v0


#####
Version: V0 (Raw Median)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/train_us.csv
Combined train + val: 439074 samples
X shape: (439074, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_1.csv ---

============================== V0 (Raw Median) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.2234042553 0.0941704036 0.1324921136       223
        Good  0.0022650155 0.8661157025 0.0045182151       605
     Average  0.9723865878 0.0042568807 0.0084766527    231625

    accuracy                      0.0065862777    232453
   macro avg  0.3993519528 0.3215143289 0.0484956605    232453
weighted avg  0.9691431510 0.0065862777 0.0085853223    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.3068237307
  Good      : 0.0619400428
  Average   : 0.0641

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V0 (Raw Median),1,0.006586,0.321514,0.399352,0.969143,0.321514,0.006586,0.048496,0.008585,0.106818,-0.042749,-0.000353
1,V0 (Raw Median),2,0.995152,0.578819,0.632182,0.995942,0.578819,0.995152,0.583477,0.995458,0.633551,0.394841,0.392393
2,V0 (Raw Median),3,0.995173,0.746878,0.565081,0.996799,0.746878,0.995173,0.629290,0.995865,0.810944,0.515825,0.497766
3,V0 (Raw Median),4,0.996748,0.839439,0.712108,0.997842,0.839439,0.996748,0.759640,0.997169,0.891899,0.660276,0.643626


## V1 (Median CDS)

In [8]:
df_v1 = run_experiment(
    base_path=base_path,
    data_folder="data_median_cdsmote",
    train_file="train_median_cdsmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V1 (Median CDS)"
)

df_v1


#####
Version: V1 (Median CDS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/train_median_cdsmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_1.csv ---

============================== V1 (Median CDS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.2500000000 0.0179372197 0.0334728033       223
        Good  0.0667399066 0.4016528926 0.1144606689       605
     Average  0.9980375531 0.9858478144 0.9919052346    231625

    accuracy                      0.9833987946    232453
   macro avg  0.4382591532 0.4684793089 0.3799462356    232453
weighted avg  0.9948960688 0.9833987946 0.9887020735    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.1339264457
  Good      : 0.6290995199
  

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V1 (Median CDS),1,0.983399,0.468479,0.438259,0.994896,0.468479,0.983399,0.379946,0.988702,0.383941,0.175028,0.135235
1,V1 (Median CDS),2,0.988234,0.541562,0.455687,0.995215,0.541562,0.988234,0.472914,0.991509,0.608667,0.235906,0.201290
2,V1 (Median CDS),3,0.996055,0.576026,0.630434,0.996420,0.576026,0.996055,0.579255,0.996150,0.635924,0.477565,0.476724
3,V1 (Median CDS),4,0.997574,0.672359,0.772506,0.997263,0.672359,0.997574,0.715685,0.997376,0.728209,0.616368,0.610083


## V2 (Median SAS)

In [9]:
df_v2 = run_experiment(
    base_path=base_path,
    data_folder="data_median_sasmote",
    train_file="train_median_sasmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V2 (Median SAS)"
)

df_v2


#####
Version: V2 (Median SAS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/train_median_sasmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_1.csv ---

============================== V2 (Median SAS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.2623574144 0.3094170404 0.2839506173       223
        Good  0.2401656315 0.1917355372 0.2132352941       605
     Average  0.9975615756 0.9979147329 0.9977381230    231625

    accuracy                      0.9951560100    232453
   macro avg  0.5000282072 0.4996891035 0.4983080115    232453
weighted avg  0.9948850127 0.9951560100 0.9950115554    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.5560202874
  Good      : 0.4375294648
  

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V2 (Median SAS),1,0.995156,0.499689,0.500028,0.994885,0.499689,0.995156,0.498308,0.995012,0.515469,0.283117,0.282726
1,V2 (Median SAS),2,0.993577,0.650438,0.494693,0.995969,0.650438,0.993577,0.546576,0.994650,0.721072,0.393025,0.373779
2,V2 (Median SAS),3,0.010363,0.443957,0.417042,0.983308,0.443957,0.010363,0.121836,0.017288,0.187687,0.077135,0.001068
3,V2 (Median SAS),4,0.994330,0.882868,0.583419,0.997480,0.882868,0.994330,0.675506,0.995557,0.926853,0.573600,0.524133


## V3 (Median Radius)

In [10]:
df_v3 = run_experiment(
    base_path=base_path,
    data_folder="data_median_radiussmote",
    train_file="train_median_radiussmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V3 (Median Radius)"
)

df_v3


#####
Version: V3 (Median Radius)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/train_median_radiussmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/test_1.csv ---

============================== V3 (Median Radius) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.1272727273 0.0313901345 0.0503597122       223
        Good  0.1457607433 0.4148760331 0.2157284057       605
     Average  0.9981229083 0.9940334593 0.9960739864    231625

    accuracy                      0.9916026035    232453
   macro avg  0.4237187930 0.4800998756 0.4207207014    232453
weighted avg  0.9950690493 0.9916026035 0.9931357436    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.1771543012
  Good     

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V3 (Median Radius),1,0.991603,0.480100,0.423719,0.995069,0.480100,0.991603,0.420721,0.993136,0.427862,0.266058,0.247534
1,V3 (Median Radius),2,0.992502,0.585875,0.491280,0.996028,0.585875,0.992502,0.500217,0.993988,0.622074,0.380574,0.350762
2,V3 (Median Radius),3,0.010252,0.455780,0.418739,0.994354,0.455780,0.010252,0.132651,0.016378,0.193323,0.105860,0.001555
3,V3 (Median Radius),4,0.994055,0.848203,0.552744,0.997385,0.848203,0.994055,0.640404,0.995375,0.906472,0.556125,0.506534


## V4 (Tree CDS)

In [11]:
df_v4 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_cdsmote",
    train_file="train_extratrees_cdsmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V4 (Tree CDS)"
)

df_v4


#####
Version: V4 (Tree CDS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/train_extratrees_cdsmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/test_1.csv ---

============================== V4 (Tree CDS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.3666666667 0.0493273543 0.0869565217       223
        Good  0.7777777778 0.0347107438 0.0664556962       605
     Average  0.9966608720 0.9999784134 0.9983168865    231625

    accuracy                      0.9965541421    232453
   macro avg  0.7137017721 0.3613388371 0.3839097015    232453
weighted avg  0.9954868154 0.9965541421 0.9950172544    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.2220885376
  Good      : 0.186

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V4 (Tree CDS),1,0.996554,0.361339,0.713702,0.995487,0.361339,0.996554,0.383910,0.995017,0.218061,0.192928,0.094606
1,V4 (Tree CDS),2,0.996309,0.465260,0.545940,0.995604,0.465260,0.996309,0.470193,0.995854,0.358557,0.405419,0.400536
2,V4 (Tree CDS),3,0.995319,0.588171,0.559062,0.996139,0.588171,0.995319,0.553733,0.995653,0.645346,0.399391,0.397840
3,V4 (Tree CDS),4,0.992979,0.914146,0.531564,0.997405,0.914146,0.992979,0.626663,0.994711,0.945834,0.545345,0.479589


## V5 (Tree SAS)

In [12]:
df_v5 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_sasmote",
    train_file="train_extratrees_sasmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V5 (Tree SAS)"
)

df_v5


#####
Version: V5 (Tree SAS)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/train_extratrees_sasmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_1.csv ---

============================== V5 (Tree SAS) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0675675676 0.0224215247 0.0336700337       223
        Good  0.4062500000 0.1289256198 0.1957340025       605
     Average  0.9971574636 0.9995769023 0.9983657171    231625

    accuracy                      0.9963734604    232453
   macro avg  0.4903250104 0.3836413489 0.4092565844    232453
weighted avg  0.9947277356 0.9963734604 0.9953512612    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.1497159404
  Good      : 0.358

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V5 (Tree SAS),1,0.996373,0.383641,0.490325,0.994728,0.383641,0.996373,0.409257,0.995351,0.289263,0.266302,0.228362
1,V5 (Tree SAS),2,0.996223,0.404246,0.476636,0.994875,0.404246,0.996223,0.427548,0.995476,0.362257,0.307549,0.289419
2,V5 (Tree SAS),3,0.989959,0.642252,0.433809,0.995710,0.642252,0.989959,0.477484,0.992597,0.712483,0.308983,0.270005
3,V5 (Tree SAS),4,0.992652,0.898948,0.522139,0.997374,0.898948,0.992652,0.614628,0.994519,0.938475,0.535077,0.467173


## V6 (Tree Radius)

In [13]:
df_v6 = run_experiment(
    base_path=base_path,
    data_folder="data_extra_trees_radiussmote",
    train_file="train_extratrees_radiussmote.csv",
    val_file="val.csv",
    test_prefix="test",
    version_name="V6 (Tree Radius)"
)

df_v6


#####
Version: V6 (Tree Radius)
#####
Loading train: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/train_extratrees_radiussmote.csv
Combined train + val: 832452 samples
X shape: (832452, 4, 20) → (samples, seq_len=4, features_per_phase)
Classes: [0 1 2] → [0 1 2]
Training done

--- Test Phase 1: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_1.csv ---

============================== V6 (Tree Radius) - Phase 1 ==============================
              precision    recall  f1-score   support

   Excellent  0.0000000000 0.0000000000 0.0000000000       223
        Good  0.1160609613 0.3272727273 0.1713543920       605
     Average  0.9976641083 0.9938823529 0.9957696400    231625

    accuracy                      0.9911939188    232453
   macro avg  0.3712416899 0.4403850267 0.3890413440    232453
weighted avg  0.9944124876 0.9911939188 0.9926686783    232453

G-Mean per class (one-vs-rest):
  Excellent : 0.0000000000
 

,Version,Phase,Accuracy,BalancedAcc,PrecMacro,PrecWeighted,RecMacro,RecWeighted,F1Macro,F1Weighted,GMean,MCC,Kappa
0,V6 (Tree Radius),1,0.991194,0.440385,0.371242,0.994412,0.440385,0.991194,0.389041,0.992669,0.000000,0.201481,0.188820
1,V6 (Tree Radius),2,0.988195,0.509172,0.372655,0.995235,0.509172,0.988195,0.396625,0.991353,0.000000,0.264211,0.222934
2,V6 (Tree Radius),3,0.954210,0.570281,0.351364,0.995652,0.570281,0.954210,0.359816,0.973695,0.536767,0.169820,0.085717
3,V6 (Tree Radius),4,0.993250,0.878564,0.536411,0.997332,0.878564,0.993250,0.628551,0.994855,0.923648,0.541888,0.481899
